# Re-ranking with a bi-encoder
The top results from BM25 or DirichletLM are re-scored with the cosine similarity between query and passage embeddings from all-MiniLM-L6-v2. To do this we need the passage text, why the stopwords and stemming index is rebuilt with the text stored as metadata

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import pandas as pd
import pyterrier as pt
from pyterrier.measures import nDCG

from helpers import (INDEX_CONFIGURATIONS, RANKING_METRICS, build_retriever, controls_for_model,
                     index_path, load_best_parameters, load_documents, load_queries_and_qrels, milliseconds_per_query,
                     open_index, save_results)

if not pt.java.started():
    pt.java.init()

/opt/anaconda3/envs/searchengines/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1019 14:44:28.070000 9115 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [2]:
documents = load_documents()
dev_queries, dev_qrels = load_queries_and_qrels("dev")
test_queries, test_qrels = load_queries_and_qrels("test")

indexer = pt.IterDictIndexer(index_path("stopwords_stemming_with_text"), overwrite=True, threads=1,
                             meta={"docno": 20, "text": 4096}, **INDEX_CONFIGURATIONS["stopwords_stemming"])
indexer.index(documents.to_dict(orient="records"))

[INFO] Opening /Users/carlson/.ir_datasets/beir/fiqa/docs.pklz4/bin with direct file access


14:44:32.088 [ForkJoinPool-1-worker-3] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (7915) - further warnings are suppressed
14:44:39.655 [ForkJoinPool-1-worker-3] WARN org.terrier.structures.indexing.Indexer -- Indexed 39 empty documents


<org.terrier.querying.IndexRef at 0x38b2e4410 jclass=org/terrier/querying/IndexRef jself=<LocalRef obj=0x30cb6c868 at 0x30d37adb0>>

In [3]:
best_parameters = load_best_parameters()
index = open_index("stopwords_stemming_with_text")
metadata_fields = ("docno", "text")

bm25 = build_retriever(index, "BM25", controls_for_model("BM25", best_parameters["BM25"]), metadata_fields)
dirichlet = build_retriever(index, "DirichletLM", controls_for_model("DirichletLM", best_parameters["DirichletLM"]),
                            metadata_fields)

In [4]:
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def score_with_bi_encoder(candidates: pd.DataFrame) -> np.ndarray:
    unique_queries = candidates["query"].unique().tolist()
    unique_query_embeddings = bi_encoder.encode(unique_queries, normalize_embeddings=True)
    embedding_of_query = dict(zip(unique_queries, unique_query_embeddings))

    query_embeddings = np.stack([embedding_of_query[query] for query in candidates["query"]])
    document_embeddings = bi_encoder.encode(candidates["text"].tolist(), normalize_embeddings=True)
    return (query_embeddings * document_embeddings).sum(axis=1)


bi_encoder_scorer = pt.apply.doc_score(score_with_bi_encoder, batch_size=128)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1704.34it/s]


## Number of re-ranked documents
Tried on 100 random dev queries, since scoring every candidate with the encoder is the slow part. Depth 100 is used below.

In [5]:
RERANK_DEPTHS = [20, 50, 100, 200]
RERANK_DEPTH = 100 # how many candidates we check when we re-rank

dev_sample = dev_queries.sample(100, random_state=0)
dev_sample_qrels = dev_qrels[dev_qrels["qid"].isin(dev_sample["qid"])]

depth_systems = {"BM25": bm25}
for depth in RERANK_DEPTHS:
    depth_systems[f"BM25 % {depth} >> bi-encoder"] = bm25 % depth >> bi_encoder_scorer

depth_results = pt.Experiment(list(depth_systems.values()), dev_sample, dev_sample_qrels, [nDCG @ 10],
                              names=list(depth_systems.keys()), round=4)
depth_results["query time (ms)"] = [round(milliseconds_per_query(system, dev_sample), 1)
                                    for system in depth_systems.values()]
depth_results

,name,nDCG@10,query time (ms)
0,BM25,0.2876,57.1
1,BM25 % 20 >> bi-encoder,0.3532,142.0
2,BM25 % 50 >> bi-encoder,0.3906,278.8
3,BM25 % 100 >> bi-encoder,0.4236,514.1
4,BM25 % 200 >> bi-encoder,0.4286,984.1


## Test results

In [6]:
systems = {
    "BM25": bm25,
    "DirichletLM": dirichlet,
    f"BM25 % {RERANK_DEPTH} >> bi-encoder": bm25 % RERANK_DEPTH >> bi_encoder_scorer,
    f"DirichletLM % {RERANK_DEPTH} >> bi-encoder": dirichlet % RERANK_DEPTH >> bi_encoder_scorer,
}

test_results = pt.Experiment(list(systems.values()), test_queries, test_qrels, RANKING_METRICS,
                             names=list(systems.keys()), round=4)
test_results["query time (ms)"] = [round(milliseconds_per_query(system, test_queries), 1)
                                   for system in systems.values()]
test_results

,name,P@10,R@100,nDCG@5,nDCG@10,nDCG@20,RR@10,query time (ms)
0,BM25,0.0702,0.5524,0.2309,0.2541,0.2788,0.3119,40.1
1,DirichletLM,0.0685,0.5412,0.2388,0.2549,0.2774,0.3199,45.3
2,BM25 % 100 >> bi-encoder,0.0966,0.5524,0.3450,0.3629,0.3835,0.4424,615.5
3,DirichletLM % 100 >> bi-encoder,0.0966,0.5412,0.3481,0.3646,0.3834,0.4458,824.0


In [7]:
save_results(test_results, "04_reranking")

## Which queries are helped and hurt
Difference in nDCG@10 between BM25 and BM25 re-ranked with the bi-encoder.

In [8]:
def ndcg_per_query(system, column_name: str) -> pd.Series:
    results = pt.Experiment([system], test_queries, test_qrels, [nDCG @ 10], perquery=True)
    return results.set_index("qid")["value"].rename(column_name)


per_query_scores = pd.concat([
    ndcg_per_query(bm25, "BM25"),
    ndcg_per_query(bm25 % RERANK_DEPTH >> bi_encoder_scorer, "re-ranked"),
], axis=1)
per_query_scores["difference"] = per_query_scores["re-ranked"] - per_query_scores["BM25"]
per_query_scores["query"] = test_queries.set_index("qid")["query"]

pd.set_option("display.max_colwidth", None)
per_query_scores.sort_values("difference", ascending=False).head(5)

,BM25,re-ranked,difference,query
qid,,,,
11088,0.0,1.0,1.0,Am I required to have a lawyer create oversee creation of my will
1297,0.0,1.0,1.0,Why aren t there solutions for electronic itemized receipt for retail in store purchases
18,0.0,1.0,1.0,1 EIN doing business under multiple business names
9824,0.0,1.0,1.0,Where can end of day data be downloaded for corporate bonds
1281,0.0,1.0,1.0,How FTB and IRS find mistakes in amended tax returns Are their processes reliable


In [9]:
per_query_scores.sort_values("difference").head(5)

,BM25,re-ranked,difference,query
qid,,,,
9737,1.000000,0.000000,-1.000000,Long 100 Short 100 investment explanation
9329,0.806574,0.000000,-0.806574,Interactive Brokers IOPTS and list of structured products
9556,1.000000,0.301030,-0.698970,How does pre market trading work
10601,1.000000,0.315465,-0.684535,Bitcoin Cost Basis Purchases
6410,0.630930,0.000000,-0.630930,Will an ETF immediately reflect a reconstitution of underlying index
